# Train langmuir 2d cnn regressor

In [ ]:
from __future__ import annotations
import argparse
import copy
import json
import random
from dataclasses import asdict, dataclass
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from scipy import stats
from torch import nn
from torch.utils.data import DataLoader, Dataset
from stochastic_fano import load_stochastic_fano, sample_spectral_images
from concentration_split import grouped_concentration_split, regression_metrics


In [ ]:
PACKAGE = Path('..')
ROOT = PROJECT = PROJECT_ROOT = PACKAGE / 'work'
SETTINGS = argparse.Namespace(data_dir=PACKAGE / 'work/database', 
                              out_dir=PACKAGE / 'work/models/cnn', 
                              figure_dir=PACKAGE / 'work/figures/cnn', 
                              train_fraction=0.7, val_fraction=0.15, test_fraction=0.15, 
                              epochs=200, batch_size=16, learning_rate=0.0001, weight_decay=1e-06, patience=50, 
                              base_channels=8, dropout=0.0, pool_height=4, pool_width=3, 
                              conv_kernel_height=3, conv_kernel_width=3, 
                              resample_training_noise=True)


In [3]:
@dataclass
class ImageNormalizer:
    min_value: float
    max_value: float

    def apply(self, image: np.ndarray) -> np.ndarray:
        image = image.astype(np.float32, copy=False)
        return (image - self.min_value) / (self.max_value - self.min_value)


In [4]:
@dataclass
class SplitIndices:
    train: np.ndarray
    val: np.ndarray
    test: np.ndarray

In [ ]:
class SpectralImageDataset(Dataset):

    def __init__(self, images: np.ndarray, metadata: pd.DataFrame, indices: np.ndarray, normalizer: ImageNormalizer) -> None:
        self.images = images
        self.metadata = metadata.reset_index(drop=True)
        self.indices = np.asarray(indices, dtype=np.int64)
        self.normalizer = normalizer
        concentration = self.metadata['concentrationM'].to_numpy(dtype=np.float32)
        self.target_log10 = np.log10(concentration).astype(np.float32)

    def __len__(self) -> int:
        return len(self.indices)

    def __getitem__(self, item: int) -> tuple[torch.Tensor, torch.Tensor]:
        index = int(self.indices[item])
        image = self.normalizer.apply(self.get_image(index))
        # X_1wt = image_wt
        x = torch.from_numpy(image[None, :, :].copy()).float()
        y = torch.tensor([self.target_log10[index]], dtype=torch.float32)
        return (x, y)

    def get_image(self, index: int) -> np.ndarray:
        return np.asarray(self.images[index], dtype=np.float32)


In [6]:
class ResampledSpectralImageDataset(SpectralImageDataset):
    """Monte Carlo acquisition noise resampled on each access."""

    def configure(self, data_dir: Path) -> None:
        self.noise_config = json.loads((data_dir / 'spectral_image_database_summary.json').read_text())
        self.fano = load_stochastic_fano(Path(self.noise_config['fano_dir']), self.noise_config['wavelength_stride'])
        self.thickness = np.load(data_dir / 'thickness_nm.npy', mmap_mode='r')
        self.epoch = 0

    def set_epoch(self, epoch: int) -> None:
        self.epoch = epoch

    def get_image(self, index: int) -> np.ndarray:
        image = sample_spectral_images(self.thickness[index:index + 1], self.fano, np.random.default_rng())[0]
        std = self.noise_config['spectrum_noise_std']
        if std > 0:
            image += np.random.default_rng().normal(scale=std, size=image.shape).astype(np.float32)
            image = np.clip(image, self.noise_config['clip_min'], self.noise_config['clip_max'])
        return image.astype(self.images.dtype).astype(np.float32)


In [7]:
class ConvStage(nn.Module):

    def __init__(self, in_channels: int, out_channels: int, kernel_size: tuple[int, int]=(3, 3), pool_size: tuple[int, int]=(2, 2)) -> None:
        super().__init__()
        padding = tuple((size // 2 for size in kernel_size))
        self.stage = nn.Sequential(nn.Conv2d(in_channels, out_channels, kernel_size=kernel_size, padding=padding), make_normalization(out_channels), nn.ReLU(inplace=True), nn.Conv2d(out_channels, out_channels, kernel_size=kernel_size, padding=padding), make_normalization(out_channels), nn.ReLU(inplace=True), nn.MaxPool2d(kernel_size=pool_size))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.stage(x)


In [8]:
def make_normalization(channels: int) -> nn.Module:
    n_groups = channels
    return nn.GroupNorm(n_groups, channels)


In [ ]:
class SpectralImageCNNRegressor(nn.Module):

    def __init__(self, base_channels: int=16, dropout: float=0.15, pooled_size: tuple[int, int]=(8, 6), conv_kernel: tuple[int, int]=(3, 3)) -> None:
        super().__init__()
        c = base_channels
        pooled_features = 8 * c * pooled_size[0] * pooled_size[1]
        self.features = nn.Sequential(ConvStage(1, c, conv_kernel, (2, 2)), ConvStage(c, 2 * c, conv_kernel, (2, 2)), ConvStage(2 * c, 4 * c, conv_kernel, (2, 2)), ConvStage(4 * c, 8 * c, conv_kernel, (2, 2)))
        self.feature_dim = pooled_features
        self.regressor = nn.Sequential(nn.AdaptiveAvgPool2d(pooled_size), nn.Flatten(), nn.Linear(pooled_features, 4 * c), nn.ReLU(inplace=True), nn.Dropout(p=dropout), nn.Linear(4 * c, 1))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # H = CNN(X), X in R^(N x C x W x T)
        features = self.features(x)
        return self.regressor(features)


In [11]:
def save_training_history(history: list[dict], run_dir: Path) -> None:
    pd.DataFrame(history).to_csv(run_dir / 'history.csv', index=False)

In [ ]:
@torch.no_grad()
def evaluate_loss(model: nn.Module, loader: DataLoader, criterion, device) -> float:
    model.eval()
    total = 0.0
    for xb, yb in loader:
        xb, yb = (xb.to(device), yb.to(device))
        prediction = model(xb)
        total += criterion(prediction.float(), yb.float()).item() * xb.size(0)
    return total / len(loader.dataset)


In [ ]:
@torch.no_grad()
def predict(model: nn.Module, loader: DataLoader, device) -> np.ndarray:
    model.eval()
    predictions = []
    for xb, _yb in loader:
        prediction = model(xb.to(device))
        predictions.append(prediction.float().cpu().numpy().reshape(-1))
    return np.concatenate(predictions)


In [ ]:
def fit_image_normalizer(images: np.ndarray, train_indices: np.ndarray) -> ImageNormalizer:
    rng = np.random.default_rng()
    sample = train_indices
    if len(sample) > 512:
        sample = rng.choice(sample, size=512, replace=False)
    values = np.asarray(images[sample], dtype=np.float32)
    min_value = float(values.min())
    max_value = float(values.max())
    return ImageNormalizer(min_value=min_value, max_value=max_value)


In [ ]:
def serializable_args(args: argparse.Namespace) -> dict:
    values = vars(args).copy()
    for key, value in values.items():
        if isinstance(value, Path):
            values[key] = str(value)
    return values


In [ ]:
args = SETTINGS
args.out_dir.mkdir(parents=True, exist_ok=True)
args.figure_dir.mkdir(parents=True, exist_ok=True)

images = np.load(args.data_dir / 'spectral_images.npy', mmap_mode='r')
metadata = pd.read_csv(args.data_dir / 'spectral_image_metadata.csv')
splits = grouped_concentration_split(metadata, args)
normalizer = fit_image_normalizer(images, splits.train)
run_metrics = []


In [ ]:
run_dir = args.out_dir
run_fig_dir = args.figure_dir
run_dir.mkdir(parents=True, exist_ok=True)
run_fig_dir.mkdir(parents=True, exist_ok=True)
figure_dir = run_fig_dir

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
datasets = {'train': SpectralImageDataset(images, metadata, splits.train, normalizer), 'val': SpectralImageDataset(images, metadata, splits.val, normalizer), 'test': SpectralImageDataset(images, metadata, splits.test, normalizer), 'train_eval': SpectralImageDataset(images, metadata, splits.train, normalizer)}

if args.resample_training_noise:
    datasets['train'] = ResampledSpectralImageDataset(images, metadata, splits.train, normalizer)
    datasets['train'].configure(args.data_dir)
loaders = {name: DataLoader(dataset, batch_size=args.batch_size, shuffle=name == 'train', pin_memory=torch.cuda.is_available()) for name, dataset in datasets.items()}
model = SpectralImageCNNRegressor(args.base_channels, args.dropout, (args.pool_height, args.pool_width), (args.conv_kernel_height, args.conv_kernel_width)).to(device)
optimizer = torch.optim.NAdam(model.parameters(), lr=args.learning_rate, weight_decay=args.weight_decay)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, factor=0.5)
criterion = nn.MSELoss()
best_state = None
best_val = float('inf')
best_epoch = 0
wait = 0
history = []
for epoch in range(1, args.epochs + 1):
    if args.resample_training_noise:
        loaders['train'].dataset.set_epoch(epoch)
    train_epoch_model = model
    train_epoch_loader = loaders['train']
    train_epoch_optimizer = optimizer
    train_epoch_criterion = criterion
    train_epoch_device = device
    train_epoch_model.train()
    train_epoch_total = 0.0
    for train_epoch_xb, train_epoch_yb in train_epoch_loader:
        train_epoch_xb, train_epoch_yb = (train_epoch_xb.to(train_epoch_device), train_epoch_yb.to(train_epoch_device))
        train_epoch_optimizer.zero_grad()
        train_epoch_prediction = train_epoch_model(train_epoch_xb)
        train_epoch_loss = train_epoch_criterion(train_epoch_prediction.float(), train_epoch_yb.float())
        train_epoch_loss.backward()
        train_epoch_optimizer.step()
        train_epoch_total += train_epoch_loss.item() * train_epoch_xb.size(0)
    train_loss = train_epoch_total / len(train_epoch_loader.dataset)
    train_eval_loss = evaluate_loss(model, loaders['train_eval'], criterion, device)
    val_loss = evaluate_loss(model, loaders['val'], criterion, device)
    scheduler.step(val_loss)
    lr = optimizer.param_groups[0]['lr']
    history.append({'epoch': epoch, 'train_loss': train_loss, 'train_eval_loss': train_eval_loss, 'val_loss': val_loss, 'learning_rate': lr})
    if val_loss < best_val - 1e-07:
        best_val = val_loss
        best_epoch = epoch
        best_state = copy.deepcopy(model.state_dict())
        torch.save({'model_state_dict': best_state, 'args': serializable_args(args), 'normalization': asdict(normalizer), 'best_epoch': best_epoch, 'best_val_loss': best_val, 'train_indices': splits.train.tolist(), 'val_indices': splits.val.tolist(), 'test_indices': splits.test.tolist()}, run_dir / f'best_epoch_{epoch:03d}.pt')
        wait = 0
    else:
        wait += 1
    save_training_history(history, run_dir)
    if epoch == 1 or epoch % 10 == 0 or wait == 0:
        print(f'epoch {epoch:03d}: train_loss={train_loss:.6g} train_eval_loss={train_eval_loss:.6g} val_loss={val_loss:.6g}')
    if wait >= args.patience:
        print(f'Early stopping at epoch {epoch}; best epoch {best_epoch}.')
        break
model.load_state_dict(best_state)
test_loss = evaluate_loss(model, loaders['test'], criterion, device)
predictions = predict(model, loaders['test'], device)
test_indices = splits.test
truth_log10 = np.log10(metadata.loc[test_indices, 'concentrationM'].to_numpy(dtype=np.float64))
metrics = regression_metrics(truth_log10, predictions)
save_training_history(history, run_dir)
pd.DataFrame({'recordId': metadata.loc[test_indices, 'recordId'].to_numpy(), 'concentrationM': metadata.loc[test_indices, 'concentrationM'].to_numpy(), 'true_log10_concentration': truth_log10, 'pred_log10_concentration': predictions, 'pred_concentrationM': np.power(10.0, predictions)}).to_csv(run_dir / 'test_predictions.csv', index=False)
torch.save({'model_state_dict': best_state, 'normalization': asdict(normalizer), 'resample_training_noise': args.resample_training_noise, 'regression_output_dtype': 'float32', 'base_channels': args.base_channels, 'dropout': args.dropout, 'norm': 'group', 'optimizer': 'nadam', 'loss': 'mse', 'pooled_size': (args.pool_height, args.pool_width), 'conv_kernel': (args.conv_kernel_height, args.conv_kernel_width), 'feature_dim': model.feature_dim, 'best_epoch': best_epoch, 'best_val_loss': best_val, 'test_loss_at_best_val_epoch': test_loss, 'train_indices': splits.train.tolist(), 'val_indices': splits.val.tolist(), 'test_indices': splits.test.tolist()}, run_dir / 'best_model.pt')


In [ ]:
output_path = figure_dir / 'training_validation_loss.svg'
frame = pd.DataFrame(history)
label = 'MSE'
fig, ax = plt.subplots(figsize=(7.0, 4.6))
ax.plot(frame['epoch'], frame['train_eval_loss'], label='Train loss (eval mode)', linewidth=1.8)
ax.plot(frame['epoch'], frame['val_loss'], label='Validation loss', linewidth=1.8)
ax.scatter([best_epoch], [best_val], color='crimson', s=36, zorder=4, label=f'Best val epoch {best_epoch}')
ax.axhline(test_loss, color='0.25', linestyle='--', linewidth=1.2, label='Held-out test loss at best val')
ax.set_title('Langmuir 2D CNN training curve')
ax.set_xlabel('Epoch')
ax.set_ylabel(f'{label} loss on log10 concentration')
ax.grid(True, alpha=0.28)
ax.legend(frameon=False)
fig.tight_layout()
#fig.savefig(output_path)
#plt.close(fig)

y_true = truth_log10
y_pred = predictions
output_path = figure_dir / 'prediction_vs_ground_truth.svg'
fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(y_true, y_pred, s=12, alpha=0.75)
lims = [min(y_true.min(), y_pred.min()), max(y_true.max(), y_pred.max())]
ax.plot(lims, lims, 'k--', linewidth=1)
ax.set_xlabel('True log10 concentration (M)')
ax.set_ylabel('Predicted log10 concentration (M)')
ax.grid(True, alpha=0.3)
fig.tight_layout()
#fig.savefig(output_path)
#plt.close(fig)
metrics = metrics
run_metrics.append({**metrics})


In [ ]:
summarize_metrics_run_metrics = run_metrics
summarize_metrics_frame = pd.DataFrame(summarize_metrics_run_metrics)
summarize_metrics_rows = []

for summarize_metrics_metric in summarize_metrics_frame.columns:
    summarize_metrics_rows.append({'metric': summarize_metrics_metric, 'mean': summarize_metrics_frame[summarize_metrics_metric].mean(), 'std': summarize_metrics_frame[summarize_metrics_metric].std()})

summary = pd.DataFrame(summarize_metrics_rows)
pd.DataFrame(run_metrics).to_csv(args.out_dir / 'test_metrics.csv', index=False)
summary.to_csv(args.out_dir / 'summary_metrics_mean_std.csv', index=False)

with (args.out_dir / 'experiment_summary.json').open('w', encoding='utf-8') as handle:
    json.dump({'args': serializable_args(args), 'image_shape': list(images.shape), 'normalization': asdict(normalizer), 'n_train': int(len(splits.train)), 'n_val': int(len(splits.val)), 'n_test': int(len(splits.test)), 'target': 'log10_concentrationM', 'metrics': summary.to_dict(orient='records')}, handle, indent=2)

print('\nAggregate test metrics:')
print(summary.to_string(index=False))
